# 특허 → 지정상품 검색 노트북

**사용법**: 아래 셀을 한 번 실행하여 인덱스를 메모리에 올린 뒤, 그 다음 셀들에서 `search(번호)` 를 호출합니다.

- 첫 셀(워밍업)은 `mmap_mode='r'` 를 써서 임베딩 파일을 가상 메모리에 매핑하므로 수 초 만에 끝납니다.
- 이후 검색은 **수십 ms** 수준.

In [1]:
import sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src.candidate import GoodsIndex, candidate_text_ids
from src.config import load_config

cfg = load_config()
cache = cfg.paths.cache_dir

patents = pd.read_parquet(cache / "patents.parquet")
patents_num_to_row = {num: i for i, num in enumerate(patents["번호"].tolist())}

goods_unique = pd.read_parquet(cache / "goods_unique.parquet")
goods_texts = goods_unique["text"].tolist()

with open(cache / "goods_index.pkl", "rb") as f:
    gi = pickle.load(f)
goods_idx = GoodsIndex(
    texts=goods_texts,
    ksic5_to_text_ids=gi["ksic5_to_text_ids"],
    all_ksic5=gi["all_ksic5"],
)

patent_emb = np.load(cache / "patent_emb.npy", mmap_mode="r")
goods_emb = np.load(cache / "goods_emb.npy", mmap_mode="r")

print(f"loaded: {len(patents):,} patents, {len(goods_texts):,} goods, dim={patent_emb.shape[1]}")

loaded: 52,238 patents, 279,680 goods, dim=1536


## 검색 함수 정의

In [2]:
def search(patent_number: str, top_k: int = 10, save_csv: bool = False, verbose: bool = True) -> pd.DataFrame:
    if patent_number not in patents_num_to_row:
        raise KeyError(f"특허번호 '{patent_number}' 를 찾을 수 없습니다.")
    row = patents_num_to_row[patent_number]
    pr = patents.iloc[row]
    title = pr["명칭"]
    ipc_list = list(pr["ipc_list"])
    ksic_prefixes = set(pr["ksic_prefixes"])

    cand_ids = candidate_text_ids(ksic_prefixes, goods_idx)
    if verbose:
        print(f"[검색] {patent_number}")
        print(f"  명칭: {title}")
        ipc_show = ', '.join(ipc_list[:8]) + (' ...' if len(ipc_list) > 8 else '')
        print(f"  IPC ({len(ipc_list)}): {ipc_show}")
        print(f"  KSIC prefix ({len(ksic_prefixes)}): {sorted(ksic_prefixes)}")
        print(f"  후보 지정상품 수: {len(cand_ids):,}")

    if len(cand_ids) == 0:
        if verbose:
            print("  후보가 없습니다 (KSIC 매칭 0건). 빈 결과 반환.")
        return pd.DataFrame(columns=["rank", "similarity", "지정상품"])

    q = patent_emb[row]
    cand_vecs = goods_emb[cand_ids]
    sims = cand_vecs @ q
    k = min(top_k, len(cand_ids))
    top_local = np.argpartition(-sims, k - 1)[:k]
    top_local = top_local[np.argsort(-sims[top_local])]
    top_ids = cand_ids[top_local]
    top_sims = sims[top_local]
    out = pd.DataFrame({
        "rank": np.arange(1, k + 1),
        "similarity": top_sims,
        "지정상품": [goods_texts[i] for i in top_ids],
    })

    if save_csv:
        safe = "".join(c if c.isalnum() else "_" for c in patent_number)
        path = cfg.paths.output_dir / f"top{top_k}_{safe}.csv"
        out.to_csv(path, index=False, encoding="utf-8-sig")
        if verbose:
            print(f"\n저장: {path}")
    return out

## 단일 특허 검색

In [6]:
search("KR20200123100A", top_k=10)

[검색] KR20200123100A
  명칭: 가변 하중을 가지는 무인비행체 비행제어 시스템 및 방법{SYSTEM AND FLIGHT CONTROL METHOD FOR UNMANNED AERIAL VEHICLE WITH VARIABLE LOAD}
  IPC (4): G05D 1/606, G05B 13/04, G06F 30/15, G05D 109/20
  KSIC prefix (2): ['272', '62']
  후보 지정상품 수: 2,238


,rank,similarity,지정상품
0,1,0.452421,항공기 조작 및 정비용 항법 장치
1,2,0.442514,가변식 속도조절기
2,3,0.429024,항공기운항 모의시험용 시뮬레이터
3,4,0.424492,항공기 조작 및 정비용 시험 장치
4,5,0.422191,항공기 조작 및 정비용 속도 검사 장치
5,6,0.408565,항공기용 비행시뮬레이터
6,7,0.406714,항공기운항 훈련요원용 시뮬레이터
7,8,0.392407,비행 훈련 시뮬레이터
8,9,0.391750,비행 제어장치
9,10,0.389226,비의료용 뇨비중계


In [7]:
search("KR20230132687A", top_k=10)

[검색] KR20230132687A
  명칭: 인공 신경망 및 검색 결과를 이용하여 악성 URL을 탐지하는 장치 및 방법{APPARTUS AND METHOD FOR DETECTING MALICIOUS URLs USING ARTIFICAIL NEURAL NETWORKS AND SEARCH RESULTS}
  IPC (5): G06F 21/56, G06F 21/57, G06F 40/205, G06N 3/045, G06N 3/08
  KSIC prefix (2): ['2631', '62']
  후보 지정상품 수: 775


,rank,similarity,지정상품
0,1,0.448789,통신네트워크 검색엔진의 개발/유지보수 및 업데이팅업
1,2,0.402950,컴퓨터 및 컴퓨터네트워크 콘텐츠용 원격검색 컴퓨터프로그램
2,3,0.390212,자동인식 시스템에 관한 기술연구업
3,4,0.386865,전자정보처리 시스템 검사업
4,5,0.382386,컴퓨터화된 행정처리의 자동화에 관한 연구업
5,6,0.380714,컴퓨터화된 자료처리업
6,7,0.372327,디지털 정보처리장치
7,8,0.370968,인터넷 보안프로그램 설계 및 개발업
8,9,0.367323,자동넘버링 시스템에 관한 기술연구업
9,10,0.366714,토론용 온라인 포럼 설계/관리 및 모니터링업


## 배치 검색 — 여러 특허 한 번에

In [5]:
patent_numbers = [
    "KR20200013719A",
    "KR20237010843A",
    "KR20257027169A",
]
results = {p: search(p, top_k=10, save_csv=False, verbose=False) for p in patent_numbers}
for p, df in results.items():
    print(f"\n=== {p} ===")
    print(df.to_string(index=False))


=== KR20200013719A ===
 rank  similarity                지정상품
    1    0.755382            반도체메모리장치
    2    0.717366              반도체메모리
    3    0.630260            반도체 기억장치
    4    0.551167          컴퓨터용 메모리카드
    5    0.539628  버퍼 메모리기기(컴퓨터 하드웨어)
    6    0.538918              메모리 장치
    7    0.534293   모바일 전기통신기기용 메모리카드
    8    0.530776              반도체 장치
    9    0.518939 보안 디지털(SD)카드용 메모리카드
   10    0.518137         스마트폰용 메모리카드

=== KR20237010843A ===
 rank  similarity             지정상품
    1    0.698740 전기식 수송기계기구용 충전장치
    2    0.679819      재충전장비용 충전기구
    3    0.664672   모터차량용 배터리 충전장치
    4    0.643146        전자담배용 충전기
    5    0.634193         축전지용 충전기
    6    0.627476      USB 자동차 충전기
    7    0.626015        자동차용 완충장치
    8    0.624777          전기충전제어기
    9    0.621442 무선이동전화기용 축전지 충전기
   10    0.616012     자동차용 구동 제어장치

=== KR20257027169A ===
 rank  similarity                      지정상품
    1    0.707139       데이터 처리 및 컴퓨터 장비 관리업
    2    0.698488              데이터 처리

## CSV 저장이 필요할 때

`save_csv=True` 로 호출하면 `output/top{K}_{번호}.csv` 에 저장됩니다.

In [ ]:
search("KR20200013719A", top_k=10, save_csv=True)